# 00 — Ambiente del taller, con el modelo en la laptop del instructor

**El modelo corre en una sola máquina y lo comparte el salón.** El backend y la
página corren en la laptop de cada quien.

Este notebook tiene **dos partes, y cada una es para una persona distinta**:

| parte | quién la corre |
|---|---|
| **Parte 1** | **solo el instructor**, en la máquina que sirve el modelo |
| **Parte 2** | **cada asistente**, en su propia laptop |

Si eres asistente, **salta directo a la Parte 2**. Correr la Parte 1 no te sirve
de nada: tu máquina no tiene el modelo descargado.

Hay otras dos versiones de este notebook:

| notebook | cuándo |
|---|---|
| `00_localhost.ipynb` | trabajas solo, todo en tu máquina |
| **`00_aula.ipynb`** | **este**: el modelo en la laptop del instructor |
| `00_tec.ipynb` | el modelo en el servidor del Tec |

## El ayudante — lo corren los dos

Define una función para correr comandos mostrando su salida en vivo. No imprime
más que la ruta del repo.

In [ ]:
import subprocess
import sys
from pathlib import Path


def raiz_del_repo():
    """
    La raiz del repo, sin importar desde donde se lanzo Jupyter.

    Puede ser notebooks/ o la raiz, segun como se abrio. Buscamos hacia arriba
    el directorio que tenga infra/ y .env.example.
    """
    p = Path.cwd().resolve()
    for candidato in (p, *p.parents):
        if (candidato / "infra").is_dir() and (candidato / ".env.example").is_file():
            return candidato
    raise RuntimeError(f"No encontre la raiz del repo desde {p}")


RAIZ = raiz_del_repo()
print("repo:", RAIZ)


def correr(comando, cwd=None, titulo=None):
    """Corre un comando e imprime su salida en vivo. Devuelve el codigo de salida."""
    if titulo:
        print(f"--- {titulo} ---")
    proceso = subprocess.Popen(
        comando,
        cwd=str(cwd or RAIZ),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,   # docker manda el progreso a stderr
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    for linea in proceso.stdout:
        print(linea.rstrip())
    codigo = proceso.wait()
    print(f"[codigo de salida: {codigo}]")
    return codigo

---

# Parte 1 — Solo el instructor

Lo que sigue corre **en la máquina que sirve el modelo, y en ninguna otra**.

No es protocolo: si veinte personas levantan Ollama, veinte máquinas intentan
cargar 6.6 GB a la vez, y el volumen de modelos solo existe en la del
instructor.

## 1.1 Levantar Ollama

Las dos celdas siguientes levantan el modelo en la GPU y esperan a que la API
conteste. La segunda espera hasta tres minutos.

In [ ]:
correr(["docker", "compose", "up", "-d"],
       cwd=RAIZ / "infra" / "aula", titulo="levantando ollama")

In [ ]:
# Esperar a que la API responda, y listar lo que tiene cargado.
import time

import requests

listo = False
for intento in range(36):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        listo = True
        break
    except Exception:
        time.sleep(5)
        print(".", end="", flush=True)
print()

if not listo:
    print("La API no respondio. Revisa:  docker logs agenticai_ollama --tail 50")
else:
    for m in r.json().get("models", []):
        print(f"  {m['name']:24} {m['size'] / 2**30:.1f} GB")

## 1.2 Abrir el firewall

Esta es la única parte que el notebook **no puede hacer solo**. Una regla de
firewall necesita privilegios de administrador y el kernel de Jupyter corre con
los tuyos normales.

Lo que sí puede hacer es **lanzar una ventana elevada**. Apruebas el diálogo de
UAC y lees el resultado ahí.

| sistema | qué pasa |
|---|---|
| **Windows** | Abre una ventana con UAC. Aprueba el diálogo. |
| **macOS** | Corre directo: el firewall filtra por aplicación. |
| **Linux** | `sudo` necesita una terminal con teclado, y Jupyter no la tiene. La celda imprime el comando para pegarlo. |

**Revisa la subred antes de correrla.** Si tu IP es `192.168.1.X`, la subred es
`192.168.1.0/24`: se cambia el último número por un cero y se agrega `/24`.

In [ ]:
SUBRED_DEL_AULA = "192.168.1.0/24"   # <-- la del aula. Si tu IP es
                                     # 192.168.1.X, la subred es esta.

if sys.platform == "win32":
    guion = RAIZ / "scripts" / "win" / "aula-firewall.ps1"
    # -NoExit deja la ventana abierta para poder leer el resultado. El script
    # detecta si la red esta marcada como publica y, en ese caso, se detiene
    # en vez de crear una regla que no aplica.
    argumentos = (f"'-NoExit','-ExecutionPolicy','Bypass','-File','{guion}',"
                  f"'-IncluirRedPublica','-Subred','{SUBRED_DEL_AULA}'")
    print("Abriendo una ventana elevada. APRUEBA EL DIALOGO DE UAC.")
    print("El resultado queda en esa ventana, no aqui.")
    correr(["powershell", "-NoProfile", "-Command",
            f"Start-Process powershell -Verb RunAs -ArgumentList {argumentos}"])

elif sys.platform == "darwin":
    correr(["bash", str(RAIZ / "scripts" / "mac" / "aula-firewall.sh")])

else:
    guion = RAIZ / "scripts" / "linux" / "aula-firewall.sh"
    print("En Linux, sudo necesita una terminal. Pega esto en una:")
    print()
    print(f"    sudo bash {guion} --subred {SUBRED_DEL_AULA}")

## 1.3 La dirección que se dicta al salón

La celda siguiente detecta la IP de LAN de esta máquina, **la escribe en el
`.env`** como `OLLAMA_URL_AULA` y la imprime.

Esa IP la asigna el DHCP y cambia sola, así que se relee el día del taller y no
se escribe a mano.

Si lista varias candidatas, la buena es la de la interfaz conectada a la red del
aula. **Descarta las de VPN**: cambian en cada reconexión.

> **Verifícalo desde OTRA laptop antes de empezar.** Desde `localhost` siempre
> funciona, y eso no prueba nada sobre lo que alcanza el salón:
> `curl http://<la-ip>:11434/api/tags`

In [ ]:
# Detecta la IP de LAN, la escribe en el .env y la imprime para dictarla.
correr([sys.executable, str(RAIZ / "scripts" / "fijar_ip_aula.py")])

---

# Parte 2 — Cada asistente, en su laptop

**Aquí empieza lo que corre todo el mundo**, incluido el instructor en su propia
máquina.

## 2.1 La configuración

La celda revisa el `.env` de la raíz y, si no coincide con lo que este notebook
necesita, dice qué línea cambiar.

La dirección del instructor —`OLLAMA_URL_AULA`— es la que él dictó en la Parte
1. Cópiala tal cual.

In [ ]:
# El .env de la raiz, que es UN solo archivo para todo: lo leen el codigo de
# src/paso*, estos notebooks y el contenedor del backend.
#
# Esta celda no edita nada: compara lo que hay contra lo que este notebook
# necesita, y si no coincide dice exactamente que cambiar.
import shutil

DESTINO_ESPERADO = "aula"
MODELO_ESPERADO = "ollama:qwen3.5:9b"

env = RAIZ / ".env"
if not env.exists():
    shutil.copy(RAIZ / ".env.example", env)
    print(f"Cree {env} desde el ejemplo.")

valores = {}
for linea in env.read_text(encoding="utf-8").splitlines():
    linea = linea.strip()
    if linea and not linea.startswith("#") and "=" in linea:
        k, v = linea.split("=", 1)
        valores[k.strip()] = v.strip()

destino = valores.get("OLLAMA_DESTINO", "")
modelo = valores.get("CHAT_MODEL", "")
url = valores.get(f"OLLAMA_URL_{destino.upper()}", "(sin declarar)")

print(f"  OLLAMA_DESTINO = {destino or '(vacio)'}   ->  {url}")
print(f"  CHAT_MODEL     = {modelo or '(vacio)'}")

faltan = []
if destino != DESTINO_ESPERADO:
    faltan.append(f"OLLAMA_DESTINO={DESTINO_ESPERADO}")
if modelo != MODELO_ESPERADO:
    faltan.append(f"CHAT_MODEL={MODELO_ESPERADO}")

if faltan:
    print(f"\n  ESTE NOTEBOOK NECESITA otra configuracion. Edita el .env de la raiz:")
    for f in faltan:
        print(f"      {f}")
    print(f"  y vuelve a correr esta celda.")
else:
    print(f"\n  Configuracion correcta para este notebook.")

### Y el inventario, desde la semilla

`datos/inventario.csv` es el **estado vivo** del dominio: el backend lo reescribe
en cada `PUT`, así que no viaja en git. En un clon nuevo no existe, y si quedó
sucio de una corrida anterior, los notebooks del 02 al 06 arrancan con números
que no son los que su texto describe.

La celda siguiente lo rehace desde
[`datos/inventario.semilla.csv`](../datos/inventario.semilla.csv) **pisando lo
que hubiera**, e imprime con qué quedó. Es Python puro: no necesita Docker ni el
backend arriba, y en eso se diferencia de
`scripts/<tu-sistema>/reiniciar-datos`, que lo pide por HTTP.

> La carpeta `datos/` está montada dentro del contenedor del backend, así que
> este archivo y el que lee el backend son el mismo.

In [ ]:
# El estado vivo, rehecho desde la semilla. En un clon nuevo no existe, y si
# quedo de una corrida anterior trae los numeros de esa corrida.
import csv
import shutil

SEMILLA = RAIZ / "datos" / "inventario.semilla.csv"
VIVO = RAIZ / "datos" / "inventario.csv"

antes = f"{VIVO.stat().st_size} bytes" if VIVO.exists() else "no existia"
shutil.copy(SEMILLA, VIVO)
print(f"  copiado: {SEMILLA.name}  ->  {VIVO.name}   (antes: {antes})")
print()

with open(VIVO, encoding="utf-8", newline="") as f:
    filas = list(csv.DictReader(f))

print(f"  {'SKU':<15} {'producto':<30} {'exist.':>7} {'precio':>7}")
print(f"  {'-' * 15} {'-' * 30} {'-' * 7} {'-' * 7}")
for r in filas:
    unidades = int(r["existencias"])
    aviso = "  <- en cero" if unidades == 0 else ""
    print(f"  {r['sku']:<15} {r['nombre']:<30} {unidades:>7} "
          f"{float(r['precio']):>7.0f}{aviso}")

print(f"\n  {len(filas)} productos. Con esto arrancan los notebooks 02 al 06.")

## 2.2 Levantar el backend y la página

Corren en tu propia laptop. Sin GPU y sin modelo: son imágenes chicas.

Local y no compartido a propósito. Si el backend fuera central, veinte personas
escribirían en el mismo tablero y cada demostración se contaminaría con la de al
lado. Lo único que se centraliza es la inferencia, que es lo único sin estado.

**La primera vez tarda unos minutos.**

In [ ]:
correr(["docker", "compose", "up", "-d", "--build"],
       cwd=RAIZ / "infra" / "taller", titulo="levantando backend y frontend")

## 2.3 Verificar

El diagnóstico completo. Para cada falla imprime qué hacer.

In [ ]:
correr([sys.executable, str(RAIZ / "scripts" / "verificar_ambiente.py")])

## 2.4 La primera llamada al modelo

Si devuelve texto, el camino completo funciona: tu laptop, la red del aula y la
máquina del instructor.

**La primera llamada del día puede tardar unos ocho segundos**, porque el modelo
se está cargando en la GPU del instructor. Esa espera la paga la primera persona
que pregunte, por todos.

In [ ]:
sys.path.insert(0, str(RAIZ / "src"))

# El .env de la raiz entra al entorno AQUI, antes de la primera llamada. Un
# kernel de Jupyter puede traer valores viejos -VS Code inyecta el .env al
# arrancarlo- y reload() no limpia variables de entorno. cargar_env() si, y
# avisa de las claves que borro por ya no estar en el archivo.
from agentlib import cargar_env

_env, _claves, _viejas = cargar_env()
print(f"  .env   : {_env}  ({_claves} claves)")
if _viejas:
    print(f"           quitadas del entorno, ya no estan en el .env: {', '.join(_viejas)}")
from agentlib import chat, ultima_medicion   # noqa: E402

respuesta = chat([{"role": "user", "content": "Responde solo con la palabra LISTO, sin nada mas."}])
print(respuesta["content"])
print()
print("  tokens enviados:", ultima_medicion()["tokens_enviados"])
print("  segundos       :", ultima_medicion()["segundos"])

## 2.5 El backend

In [ ]:
import requests

print("salud  :", requests.get("http://localhost:8000/api/health", timeout=5).json())
print("modelos:")
for m in requests.get("http://localhost:8000/api/modelos", timeout=15).json()["modelos"]:
    print(f"  {'listo' if m['listo'] else '  -  '}  {m['id']}")

Y la página: **http://localhost:4200**

Déjala abierta en otra ventana. Desde el paso 3 en adelante es donde se ve el
efecto de lo que hace el agente.

---

# Al terminar

**Cada asistente** baja lo suyo con la celda siguiente.

In [ ]:
correr(["docker", "compose", "down"],
       cwd=RAIZ / "infra" / "taller", titulo="bajando backend y frontend")

## Y solo el instructor: bajar Ollama y cerrar el firewall

**Nunca uses `down -v`.** El volumen de modelos es externo y lo comparte el
proyecto itzae: borraría los 10 GB de los dos.

La regla de firewall **no se baja sola**: persiste hasta que la quites.

In [ ]:
correr(["docker", "compose", "down"],
       cwd=RAIZ / "infra" / "aula", titulo="bajando ollama")

In [ ]:
if sys.platform == "win32":
    guion = RAIZ / "scripts" / "win" / "aula-firewall.ps1"
    argumentos = f"'-NoExit','-ExecutionPolicy','Bypass','-File','{guion}','-Quitar'"
    print("Abriendo una ventana elevada. APRUEBA EL DIALOGO DE UAC.")
    correr(["powershell", "-NoProfile", "-Command",
            f"Start-Process powershell -Verb RunAs -ArgumentList {argumentos}"])

elif sys.platform == "darwin":
    correr(["bash", str(RAIZ / "scripts" / "mac" / "aula-firewall.sh"), "--quitar"])

else:
    guion = RAIZ / "scripts" / "linux" / "aula-firewall.sh"
    print("En Linux, pega esto en una terminal:")
    print()
    print(f"    sudo bash {guion} --quitar")

In [ ]:
# Que no quedo nada arriba.
correr(["docker", "ps", "--filter", "name=agenticai",
        "--format", "{{.Names}}  {{.Status}}"], titulo="contenedores que siguen vivos")
print("(si no lista ninguno, esta todo abajo)")